# Aperture -- Attention Rollout (Kaggle GPU worker)
OpenVLA-7B in 4-bit on a free Kaggle P100/T4, attention rollout over self-attention,
plus a scaffold to poll the Aperture API for queued attribution jobs.

**Sidebar -> Settings -> Accelerator: GPU P100 - Internet: On** before running.

In [ ]:
import torch, socket
assert torch.cuda.is_available(), "No GPU -- Settings -> Accelerator -> GPU P100"
try:
    socket.create_connection(("huggingface.co", 443), timeout=5)
except OSError:
    raise RuntimeError("No internet -- Settings -> Internet -> On")
p = torch.cuda.get_device_properties(0)
print(torch.cuda.get_device_name(0), f"{p.total_memory/1e9:.1f} GB")

### Install pinned deps
Kaggle ships a newer `transformers`; these pins downgrade it to the stack OpenVLA expects.

> After this runs, do **Run -> Restart & clear cell outputs** once, then continue -- Kaggle keeps the old `transformers` in memory until a kernel restart.

In [ ]:
!pip install -q "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.16" \
    accelerate bitsandbytes pillow requests numpy boto3

### Secrets (Kaggle Secrets add-on)
Add-ons -> **Secrets** -> add `APERTURE_API_KEY` (and `HF_TOKEN` / R2 creds if used).

In [ ]:
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()

def secret(name, default=None):
    try: return sec.get_secret(name)
    except Exception: return default

API_BASE = secret("APERTURE_API_BASE", "https://aperture-api.onrender.com")
API_KEY  = secret("APERTURE_API_KEY", "demo-key")
MODEL_ID = secret("APERTURE_VLA_MODEL", "openvla/openvla-7b")

hf = secret("HF_TOKEN")
if hf:
    from huggingface_hub import login; login(hf)   # only if you hit a gated/401 error

### Load model (4-bit + eager attention)

In [ ]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb,        # ~5 GB instead of ~15 GB; fits a free T4/P100
    attn_implementation="eager",    # REQUIRED -- flash/SDPA return attentions=None
    output_attentions=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map={"": 0},
).eval()
print("loaded")

### Rollout (unchanged patent-precise method)

In [ ]:
import numpy as np, torch

def attention_rollout(attentions):
    """attentions: list of [batch, heads, tokens, tokens] per layer -> query->patch map."""
    result = None
    for attn in attentions:
        a = attn.mean(dim=1)[0]
        a = 0.5 * a + 0.5 * torch.eye(a.size(0), device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        result = a if result is None else a @ result
    q = result[0, 1:]
    g = int(np.sqrt(q.shape[0]))
    return q[: g * g].reshape(g, g).detach().cpu().numpy()

### Smoke test on one frame

In [ ]:
import requests, torch
from PIL import Image

img = Image.open(requests.get(
    "http://images.cocodataset.org/val2017/000000039769.jpg", stream=True).raw).convert("RGB")
prompt = "In: What action should the robot take to pick up the object?\nOut:"

inputs = processor(prompt, img).to(0, dtype=torch.float16)
with torch.no_grad():
    out = model(**inputs, output_attentions=True)

assert out.attentions is not None, "attentions is None -- attn_implementation must be 'eager'"
heat = attention_rollout(out.attentions)
print("attn layers:", len(out.attentions), "| heatmap grid:", heat.shape)

import matplotlib.pyplot as plt
plt.imshow(heat, cmap="inferno"); plt.title("attention rollout"); plt.colorbar(); plt.show()

### R2 + API clients (if credentials are set as secrets)

In [ ]:
import boto3
r2 = boto3.client('s3', endpoint_url=secret('APERTURE_R2_ENDPOINT_URL'),
                  aws_access_key_id=secret('APERTURE_R2_ACCESS_KEY_ID'),
                  aws_secret_access_key=secret('APERTURE_R2_SECRET_ACCESS_KEY'))
BUCKET = secret('APERTURE_R2_BUCKET', 'aperture-blobs')
HDRS = {'X-API-Key': API_KEY}

### Episode processing + poll loop scaffold

In [ ]:
import io, json, time, requests, numpy as np, torch
from PIL import Image

def process_episode_frames(frames, instruction="complete the task"):
    seq = []
    prompt = f"In: What action should the robot take to {instruction}?\nOut:"
    for t, img in enumerate(frames):
        inputs = processor(prompt, img.convert("RGB")).to(0, dtype=torch.float16)
        with torch.no_grad():
            out = model(**inputs, output_attentions=True)
        heat = attention_rollout(out.attentions)
        heat = (heat - heat.min()) / (np.ptp(heat) + 1e-8)
        seq.append({'t': t, 'grid': np.round(heat, 5).tolist()})
    return {'simulated': False, 'grid_size': len(seq[0]['grid']) if seq else 0, 'frames': seq}

def poll_once():
    jobs = requests.get(f"{API_BASE}/v1/attribution_jobs?status=queued", headers=HDRS).json()
    for job in jobs:
        # TODO: fetch episode, decode frames from its rlds_uri blob, then:
        # result = process_episode_frames(frames, job['instruction'])
        # key = f"rollouts/{job['id']}.json"
        # r2.put_object(Bucket=BUCKET, Key=key, Body=json.dumps(result).encode())
        # requests.post(f"{API_BASE}/v1/attribution_jobs/{job['id']}/done",
        #               headers=HDRS, json={'result_uri': key})
        pass
    return len(jobs)

print("Worker ready. Uncomment the poll loop when API + R2 creds are set.")
# while True:
#     n = poll_once(); print("processed", n); time.sleep(10)